In [ ]:
"""
CineMatch — Full Catalog FAISS Rebuild Pipeline
================================================
Workflow:
    1.  [2022→today]  Fetch all movies via TMDB Discover API (target languages)
    2.  [2022→today]  Fetch full movie details (returns imdb_id + metadata)
    3.  [ALL rows]    Merge fresh imdb_rating/imdb_votes from title.ratings.tsv
    4.  [ALL rows]    Rebuild best_rating / best_votes and movieDoc text
    5.  [ALL rows]    Save updated catalog back in-place to the same CSV
    6.  [ALL rows]    Encode with BGE-M3 and rebuild FAISS from scratch
    7.              Upload CSV, FAISS index, and manifest to HF Dataset

Outputs (all relative to cinematch/):
    Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv
    outputs/tmdb/bge/tmdb_bge_m3_flatip.faiss
    outputs/tmdb/bge/tmdb_bge_m3_build_manifest.json
"""

!pip install sentence-transformers faiss-gpu-cu12 pandas numpy requests python-dotenv

from __future__ import annotations

import json
import os
import time
from datetime import date, timedelta
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# ━━━━━━━━━━━━━━━━━━━━━  CONFIG  ━━━━━━━━━━━━━━━━━━━━━━

MODEL_ID        = "BAAI/bge-m3"

# Languages to fetch from TMDB Discover API (2022 → today)
TARGET_LANGS    = ["en", "te", "ja", "ko", "hi", "ta", "ml", "kn"]

# Fetch window: ALL movies released from this date to today get fresh data
FETCH_START_DATE = "2022-01-01"

# Encoding batch size (reduce to 512 if OOM)
ENCODE_BATCH    = 512

# TMDB API config
MAX_DISCOVER_PAGES = 500   # TMDB hard cap -- do NOT raise above 500
WINDOW_MONTHS      = 1     # size of each date chunk (months per language window)
DETAIL_BATCH_LOG   = 500   # print progress every N details
MAX_WORKERS        = 20    # parallel detail fetchers

# IMDb TSV reader chunk size
IMDB_CHUNK_SIZE = 1_000_000

# ━━━━━━━━━━━━━━━━━━━━━  PATHS  ━━━━━━━━━━━━━━━━━━━━━━━

def detect_paths() -> dict:
    """Auto-detect runtime and resolve all file paths.
    
    On Colab: base = /content/drive/MyDrive/cinematch
    On HPC:   base = /blue/egn6933/nagabhairava.r
    On Local: walks up from cwd looking for Data/ + src/
    
    All outputs are relative to cinematch/ root.
    CSV is overwritten in-place (same filename, no separate updated_ file).
    """
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cinematch")
        print("Runtime: Colab")
    except ImportError:
        hpc = Path("/blue/egn6933/nagabhairava.r")
        if hpc.exists():
            base = hpc
            print("Runtime: HPC")
        else:
            here = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
            for candidate in [here, *here.parents]:
                if (candidate / "Data").exists() and (candidate / "src").exists():
                    base = candidate
                    break
            else:
                base = Path.cwd()
            print("Runtime: Local")

    # Single canonical CSV path — overwritten in-place every run
    catalog = base / "Data" / "tmdb_semantic_catalog_alllangs_with_new_movies.csv"

    # FAISS output dir: outputs/tmdb/bge/ relative to cinematch root
    out_dir = base / "outputs" / "tmdb" / "bge"

    return {
        "base":          base,
        # One CSV — same file read in, same file written out
        "catalog":       catalog,
        # FAISS artifacts
        "out_dir":       out_dir,
        "faiss":         out_dir / "tmdb_bge_m3_flatip.faiss",
        "manifest":      out_dir / "tmdb_bge_m3_build_manifest.json",
        # IMDb source
        "imdb_tsv":      base / "Data" / "title.ratings.tsv",
    }


In [ ]:

# ━━━━━━━━━━━━━━━━━━  TMDB API HELPERS  ━━━━━━━━━━━━━━━━━

def setup_tmdb_auth() -> dict:
    """Load TMDB bearer token from environment or Colab secrets."""
    load_dotenv()
    token = os.environ.get("TMDB_BEARER_TOKEN") or os.environ.get("TMDB_API_KEY")
    if not token:
        try:
            from google.colab import userdata
            try:
                token = userdata.get("TMDB_BEARER_TOKEN")
            except Exception:
                token = userdata.get("TMDB_API_KEY")
        except (ImportError, Exception):
            pass
    assert token, (
        "Set TMDB_BEARER_TOKEN in your .env file, environment, or Colab Secrets.\n"
        "Get one at https://www.themoviedb.org/settings/api"
    )
    return {"accept": "application/json", "Authorization": f"Bearer {token}"}


TMDB_BASE = "https://api.themoviedb.org/3"


def tmdb_get(url: str, headers: dict, params: dict | None = None,
             max_retries: int = 8, sleep_base: float = 1.0) -> dict:
    """TMDB GET with exponential backoff for rate limits and server errors."""
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                wait = float(r.headers.get("Retry-After", sleep_base * (2 ** attempt)))
                time.sleep(wait)
                continue
            if r.status_code in (500, 502, 503, 504):
                time.sleep(sleep_base * (2 ** attempt))
                continue
            raise RuntimeError(f"TMDB {r.status_code}: {r.text[:200]}")
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(sleep_base * (2 ** attempt))
    raise RuntimeError("Max retries exceeded")


def discover_movies(headers: dict, lang_code: str,
                    start_date: str, end_date: str | None = None,
                    page: int = 1) -> dict:
    if end_date is None:
        end_date = date.today().isoformat()
    return tmdb_get(
        f"{TMDB_BASE}/discover/movie",
        headers=headers,
        params={
            "include_adult": "true",
            "include_video": "false",
            # sort_by asc makes windowed pagination deterministic:
            # each window sees the same movies regardless of new additions.
            "sort_by": "primary_release_date.asc",
            "page": page,
            "primary_release_date.gte": start_date,
            "primary_release_date.lte": end_date,
            "with_original_language": lang_code,
            "language": "en-US",
            "with_runtime.gte": 60,
        },
    )


def make_monthly_windows(start: str, end: str) -> list[tuple[str, str]]:
    """
    Split [start, end] into (window_start, window_end) tuples of WINDOW_MONTHS
    months each. The last window extends to `end` regardless of its length.
    """
    import calendar

    def parse(d: str) -> date:
        return date.fromisoformat(d)

    def add_months(d: date, n: int) -> date:
        month = d.month - 1 + n
        year  = d.year + month // 12
        month = month % 12 + 1
        day   = min(d.day, calendar.monthrange(year, month)[1])
        return date(year, month, day)

    windows = []
    cur = parse(start)
    end_dt = parse(end)
    while cur < end_dt:
        nxt     = add_months(cur, WINDOW_MONTHS)
        win_end = min(nxt - timedelta(days=1), end_dt)
        windows.append((cur.isoformat(), win_end.isoformat()))
        cur = nxt
    return windows


def fetch_discover_window(headers: dict, lang_code: str,
                          start_date: str, end_date: str) -> list[dict]:
    """Fetch all Discover results for one language within ONE date window (<=500 pages)."""
    results = []
    page = 1
    total_pages = 1
    while page <= min(total_pages, MAX_DISCOVER_PAGES):
        data = discover_movies(headers, lang_code, start_date, end_date, page)
        batch = data.get("results", [])
        if not batch:
            break
        results.extend(batch)
        total_pages = data.get("total_pages", 1)
        if page >= total_pages:
            break
        page += 1
    if total_pages > MAX_DISCOVER_PAGES:
        print(
            f"  WARNING [{lang_code}] {start_date}->{end_date}: "
            f"total_pages={total_pages} > cap {MAX_DISCOVER_PAGES}. "
            f"Only {len(results)} / {total_pages * 20} fetched. "
            "Reduce WINDOW_MONTHS."
        )
    return results


def fetch_discover_all_windowed(headers: dict, lang_code: str,
                                start_date: str, end_date: str) -> list[dict]:
    """
    Discover all movies for one language from start_date to end_date using
    WINDOW_MONTHS-sized sub-windows to stay under TMDB's 500-page cap.
    """
    windows = make_monthly_windows(start_date, end_date)
    all_results: list[dict] = []
    seen_ids: set[int] = set()

    for i, (w_start, w_end) in enumerate(windows):
        batch = fetch_discover_window(headers, lang_code, w_start, w_end)
        new_in_batch = 0
        for r in batch:
            rid = r.get("id")
            if rid and rid not in seen_ids:
                seen_ids.add(rid)
                all_results.append(r)
                new_in_batch += 1
        # Log every 6 windows (~6 months) or first/last
        if i == 0 or (i + 1) % 6 == 0 or i == len(windows) - 1:
            print(
                f"    [{lang_code}] window {i+1}/{len(windows)}  "
                f"{w_start}->{w_end}: +{new_in_batch} "
                f"(total so far: {len(all_results):,})"
            )
    return all_results


def fetch_movie_details(headers: dict, tmdb_id: int) -> dict:
    """Fetch full movie details including keywords and imdb_id."""
    return tmdb_get(
        f"{TMDB_BASE}/movie/{tmdb_id}",
        headers=headers,
        params={
            "language": "en-US",
            "include_adult": "true",
            "append_to_response": "keywords",
        },
    )


In [3]:

# ━━━━━━━━━━━━━━━━  IMDB + MOVIEDOC HELPERS  ━━━━━━━━━━━━

def extract_keyword_names(keywords_obj) -> list[str]:
    """Extract keyword names from TMDB keywords response."""
    if isinstance(keywords_obj, dict):
        items = keywords_obj.get("keywords", []) or keywords_obj.get("results", [])
    elif isinstance(keywords_obj, list):
        items = keywords_obj
    else:
        return []
    return [k.get("name", "").strip() for k in items if isinstance(k, dict) and k.get("name")]


def extract_spoken_language_names(spoken_obj) -> list[str]:
    """Extract spoken language names from TMDB spoken_languages field."""
    if isinstance(spoken_obj, list):
        return [
            (s.get("english_name") or s.get("name") or "").strip()
            for s in spoken_obj
            if isinstance(s, dict) and (s.get("english_name") or s.get("name"))
        ]
    return []


def load_imdb_ratings_subset(ratings_tsv: Path, imdb_ids: set[str]) -> pd.DataFrame:
    """Scan title.ratings.tsv in chunks and return rows for imdb_ids."""
    empty = pd.DataFrame(columns=["tconst", "imdb_rating", "imdb_votes"])
    if not imdb_ids or not ratings_tsv.exists():
        if not ratings_tsv.exists():
            print(f"  ⚠ IMDb TSV not found: {ratings_tsv}")
        return empty

    print(f"  Scanning IMDb ratings TSV for {len(imdb_ids):,} IDs...")
    matches = []
    for chunk in pd.read_csv(
        ratings_tsv,
        sep="\t",
        usecols=["tconst", "averageRating", "numVotes"],
        dtype={"tconst": "string", "averageRating": "float32", "numVotes": "Int64"},
        chunksize=IMDB_CHUNK_SIZE,
        low_memory=False,
    ):
        picked = chunk[chunk["tconst"].isin(imdb_ids)]
        if not picked.empty:
            matches.append(picked)

    if not matches:
        return empty

    imdb = pd.concat(matches, ignore_index=True)
    imdb = imdb.rename(columns={"averageRating": "imdb_rating", "numVotes": "imdb_votes"})
    imdb["imdb_rating"] = pd.to_numeric(imdb["imdb_rating"], errors="coerce")
    imdb["imdb_votes"] = pd.to_numeric(imdb["imdb_votes"], errors="coerce").fillna(0).astype(int)
    imdb = imdb.dropna(subset=["tconst"]).drop_duplicates(subset=["tconst"])
    print(f"  Matched {len(imdb):,} IMDb rating rows.")
    return imdb


def build_moviedoc(row: dict) -> str:
    """Build movieDoc string from a row dict (works for both new TMDB rows and existing CSV rows)."""
    title          = str(row.get("title", "") or "").strip()
    original_title = str(row.get("original_title", "") or "").strip()
    overview       = str(row.get("overview", "") or "").strip()
    tagline        = str(row.get("tagline", "") or "").strip()
    lang           = str(row.get("original_language", "") or "").strip()
    imdb_id        = str(row.get("imdb_id", "") or "").strip()

    release_date = str(row.get("release_date", "") or "").strip()
    year = str(row.get("year", "") or "").strip()
    if not year:
        year = release_date[:4] if release_date and len(release_date) >= 4 and release_date[:4].isdigit() else ""

    vote_average = row.get("vote_average")
    vote_count   = row.get("vote_count")
    imdb_rating  = row.get("imdb_rating")
    imdb_votes   = row.get("imdb_votes")
    best_rating  = row.get("best_rating")
    best_votes   = row.get("best_votes")
    popularity   = row.get("popularity")

    # Genres: accept both list-of-dicts (TMDB response) and comma string (CSV)
    genres = row.get("genres", [])
    if isinstance(genres, list):
        genres_list = [g.get("name") for g in genres if isinstance(g, dict) and g.get("name")]
    else:
        genres_list = [x.strip() for x in str(genres).split(",") if x.strip()] if pd.notna(genres) else []

    spoken_list   = extract_spoken_language_names(row.get("spoken_languages", []))
    keywords_list = extract_keyword_names(row.get("keywords", {}))

    # spoken_languages in CSV is a plain string — handle that too
    if not spoken_list and isinstance(row.get("spoken_languages"), str):
        raw = row.get("spoken_languages", "")
        if pd.notna(raw) and raw:
            spoken_list = [x.strip() for x in str(raw).split(",") if x.strip()]

    # keywords in CSV is a plain string
    if not keywords_list and isinstance(row.get("keywords"), str):
        raw = row.get("keywords", "")
        if pd.notna(raw) and raw:
            keywords_list = [x.strip() for x in str(raw).split(",") if x.strip()]

    lines = [
        f"Title: {title}",
        f"Original title: {original_title}" if original_title and original_title != title else None,
        f"Release date: {release_date}" if release_date else None,
        f"Year: {year}" if year else None,
        f"IMDb id: {imdb_id}" if imdb_id else None,
        f"Original language: {lang}" if lang else None,
        f"Spoken languages: {', '.join(spoken_list[:5])}" if spoken_list else None,
        f"Genres: {', '.join(genres_list[:6])}" if genres_list else None,
        f"IMDb rating: {float(imdb_rating):.2f}" if pd.notna(imdb_rating) and imdb_rating else None,
        f"IMDb votes: {int(imdb_votes)}" if pd.notna(imdb_votes) and imdb_votes else None,
        f"Best rating: {float(best_rating):.2f}" if pd.notna(best_rating) and best_rating else None,
        f"Best votes: {int(best_votes)}" if pd.notna(best_votes) and best_votes else None,
        f"Vote average: {float(vote_average):.2f}" if pd.notna(vote_average) and vote_average else None,
        f"Vote count: {int(vote_count)}" if pd.notna(vote_count) and vote_count else None,
        f"Popularity: {float(popularity):.2f}" if pd.notna(popularity) and popularity else None,
        f"Keywords: {', '.join(keywords_list[:12])}" if keywords_list else None,
        f"Tagline: {tagline}" if tagline else None,
        f"Plot: {overview}" if overview else None,
    ]
    return "\n".join([x for x in lines if x])


def flatten_genres(g) -> str:
    if isinstance(g, list):
        return ", ".join(x.get("name", "") for x in g if isinstance(x, dict))
    return str(g) if pd.notna(g) else ""


def flatten_list_field(raw, extractor_fn) -> str:
    names = extractor_fn(raw)
    return ", ".join(names)


In [4]:
import torch
import os
os.environ["PYTORCH_NO_CUDA_MEMORY_CACHING"] = "1"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━  STEP 1 & 2: FETCH and REFRESH ALL IMDB  ━━━━━

def main():
    """
    Step 1: Fetch all movies 2022→today from TMDB Discover API.
    Step 2: Merge fresh IMDb ratings from title.ratings.tsv for the fetched movies.
    Step 3: Upsert 2022→today rows into the catalog CSV.
    Step 4: For ALL rows (including pre-2022), refresh IMDb data from TSV and rebuild movieDoc.
    Step 5: Save the catalog CSV in-place.
    """
    import concurrent.futures

    t_start = time.time()
    paths   = detect_paths()
    paths["out_dir"].mkdir(parents=True, exist_ok=True)
    headers = setup_tmdb_auth()

    today       = date.today().isoformat()
    fetch_start = FETCH_START_DATE
    print(f"\n{'═'*60}")
    print(f"  Fetch window: {fetch_start} → {today}")
    print(f"{'═'*60}\n")

    # ── 1. Discover all movies in the fetch window ─────────────────
    print(f"{'─'*60}")
    print(" Discovering movies from TMDB Discover API")
    print(f"{'─'*60}")

    all_discover = []
    for lang in TARGET_LANGS:
        n_windows = len(make_monthly_windows(fetch_start, today))
        print(f"  Discovering [{lang}] using {n_windows} monthly windows...", flush=True)
        results = fetch_discover_all_windowed(headers, lang, start_date=fetch_start, end_date=today)
        print(f"  [{lang}] total: {len(results):,}")
        all_discover.extend(results)

    discover_df = pd.DataFrame(all_discover).drop_duplicates(subset=["id"])
    discover_df["id"] = pd.to_numeric(discover_df["id"], errors="coerce")
    active_ids = [int(x) for x in discover_df["id"].dropna().unique()]
    print(f"\n  Total unique IDs to refresh: {len(active_ids):,}")

    if not active_ids:
        print("  No movies found. Exiting.")
        return

    # ── 2. Fetch full movie details (parallel) ─────────────────────
    print(f"\n{'─'*60}")
    print(f" Fetching full details for {len(active_ids):,} movies ({MAX_WORKERS} workers)")
    print(f"{'─'*60}")

    details_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_id = {}
        for tid in active_ids:
            future = executor.submit(fetch_movie_details, headers, tid)
            future_to_id[future] = tid
            time.sleep(1.0 / 35.0)  # ~35 req/s to stay within TMDB limits

        for i, future in enumerate(concurrent.futures.as_completed(future_to_id), 1):
            tid = future_to_id[future]
            try:
                details_list.append(future.result())
            except Exception as e:
                print(f"\n  [Error] ID {tid}: {e}")
            if i % DETAIL_BATCH_LOG == 0 or i == len(active_ids):
                print(f"  Fetched {i:,} / {len(active_ids):,}", flush=True)

    print(f"\n  Successfully fetched {len(details_list):,} movie details.")

    # ── 3. Process fetched movies → merge IMDb ratings ─────────────
    new_movies = pd.DataFrame(details_list)
    new_movies["id"] = pd.to_numeric(new_movies["id"], errors="coerce")
    new_movies = new_movies.dropna(subset=["id"])
    new_movies["id"] = new_movies["id"].astype(int)

    # Extract year from release_date
    new_movies["release_date"] = new_movies["release_date"].fillna("").astype(str)
    new_movies["year"] = new_movies["release_date"].str.slice(0, 4).where(
        new_movies["release_date"].str.match(r"^\d{4}", na=False), other=""
    )

    # Merge fresh IMDb ratings for the 2022→today batch
    imdb_ids_new = set(
        new_movies["imdb_id"].dropna().astype(str).str.strip()
        .loc[lambda s: s.str.startswith("tt")].tolist()
    )
    print(f"\n  Movies with imdb_id: {len(imdb_ids_new):,}")
    imdb_new = load_imdb_ratings_subset(paths["imdb_tsv"], imdb_ids_new)

    if not imdb_new.empty:
        new_movies = new_movies.drop(columns=["imdb_rating", "imdb_votes"], errors="ignore")
        new_movies = new_movies.merge(imdb_new, left_on="imdb_id", right_on="tconst", how="left")
        new_movies = new_movies.drop(columns=["tconst"], errors="ignore")
        print(f"  IMDb matches (new batch): {new_movies['imdb_rating'].notna().sum():,} / {len(new_movies):,}")
    else:
        new_movies["imdb_rating"] = np.nan
        new_movies["imdb_votes"]  = 0

    # Derive best_* columns (prefer IMDb over TMDB vote_average)
    new_movies["vote_average"] = pd.to_numeric(new_movies.get("vote_average"), errors="coerce")
    new_movies["vote_count"]   = pd.to_numeric(new_movies.get("vote_count"), errors="coerce").fillna(0)
    new_movies["imdb_rating"]  = pd.to_numeric(new_movies["imdb_rating"], errors="coerce")
    new_movies["imdb_votes"]   = pd.to_numeric(new_movies["imdb_votes"], errors="coerce").fillna(0).astype(int)
    new_movies["best_rating"]  = new_movies["imdb_rating"].fillna(new_movies["vote_average"])
    new_movies["best_votes"]   = new_movies["imdb_votes"].where(
        new_movies["imdb_votes"] > 0, new_movies["vote_count"]
    )
    new_movies["best_votes"]   = pd.to_numeric(new_movies["best_votes"], errors="coerce").fillna(0).astype(int)

    # Build movieDoc for new/updated rows
    print("  Building movieDoc for new/updated rows...")
    new_movies["movieDoc"] = new_movies.apply(lambda row: build_moviedoc(row.to_dict()), axis=1)

    # Flatten list fields to CSV-friendly strings
    KEEP_COLS = [
        "id", "title", "vote_average", "vote_count", "status", "release_date",
        "revenue", "runtime", "adult", "backdrop_path", "budget", "homepage",
        "imdb_id", "original_language", "original_title", "overview", "popularity",
        "poster_path", "tagline", "genres", "production_companies",
        "production_countries", "spoken_languages", "keywords", "year",
        "movieDoc", "best_rating", "best_votes", "imdb_rating", "imdb_votes",
    ]
    for col in KEEP_COLS:
        if col not in new_movies.columns:
            new_movies[col] = np.nan

    new_for_catalog = new_movies[KEEP_COLS].copy()
    new_for_catalog["genres"]            = new_for_catalog["genres"].apply(flatten_genres)
    new_for_catalog["spoken_languages"]  = new_for_catalog["spoken_languages"].apply(
        lambda x: flatten_list_field(x, extract_spoken_language_names)
    )
    new_for_catalog["keywords"]          = new_for_catalog["keywords"].apply(
        lambda x: flatten_list_field(x, extract_keyword_names)
    )
    new_for_catalog["production_companies"] = new_for_catalog["production_companies"].apply(
        lambda x: flatten_genres(x) if isinstance(x, list) else str(x) if pd.notna(x) else ""
    )
    new_for_catalog["production_countries"] = new_for_catalog["production_countries"].apply(
        lambda x: flatten_genres(x) if isinstance(x, list) else str(x) if pd.notna(x) else ""
    )

    # ── 4. Upsert new/updated rows into the existing catalog CSV ───
    print(f"\n{'─'*60}")
    print(" Upserting into catalog CSV")
    print(f"{'─'*60}")

    if paths["catalog"].exists():
        print(f"  Loading existing catalog: {paths['catalog'].name}")
        df_catalog = pd.read_csv(paths["catalog"], low_memory=False)
        df_catalog["id"] = pd.to_numeric(df_catalog["id"], errors="coerce")
        df_catalog = df_catalog.dropna(subset=["id"])
        df_catalog["id"] = df_catalog["id"].astype(int)

        new_for_catalog["id"] = pd.to_numeric(new_for_catalog["id"], errors="coerce")
        new_for_catalog = new_for_catalog.dropna(subset=["id"])
        new_for_catalog["id"] = new_for_catalog["id"].astype(int)

        df_catalog     = df_catalog.drop_duplicates(subset=["id"], keep="last").set_index("id")
        new_for_catalog = new_for_catalog.drop_duplicates(subset=["id"], keep="last").set_index("id")

        # Add any new columns that don't yet exist in old catalog
        for col in new_for_catalog.columns:
            if col not in df_catalog.columns:
                df_catalog[col] = np.nan

        # Overwrite existing rows and append truly new IDs
        df_catalog.update(new_for_catalog)
        new_ids = new_for_catalog.index.difference(df_catalog.index)
        if len(new_ids) > 0:
            df_catalog = pd.concat([df_catalog, new_for_catalog.loc[new_ids]])

        df_catalog = df_catalog.reset_index()
        print(f"  Existing rows: {len(df_catalog) - len(new_ids):,} | New rows added: {len(new_ids):,}")
    else:
        df_catalog = new_for_catalog.reset_index()
        print("  No existing catalog found — starting fresh.")

    print(f"  Total rows after upsert: {len(df_catalog):,}")

    # ── 5. Refresh ALL IMDb ratings from TSV (pre-2022 rows too) ───
    print(f"\n{'─'*60}")
    print(" Refreshing ALL rows' IMDb ratings from title.ratings.tsv")
    print(f"{'─'*60}")

    all_imdb_ids = set(
        df_catalog["imdb_id"].dropna().astype(str).str.strip()
        .loc[lambda s: s.str.startswith("tt")].tolist()
    )
    print(f"  Total rows with imdb_id: {len(all_imdb_ids):,}")

    imdb_all = load_imdb_ratings_subset(paths["imdb_tsv"], all_imdb_ids)

    if not imdb_all.empty:
        df_catalog = df_catalog.drop(columns=["imdb_rating", "imdb_votes"], errors="ignore")
        df_catalog = df_catalog.merge(imdb_all, left_on="imdb_id", right_on="tconst", how="left")
        df_catalog = df_catalog.drop(columns=["tconst"], errors="ignore")
        matched = df_catalog["imdb_rating"].notna().sum()
        print(f"  Matched {matched:,} / {len(df_catalog):,} rows with IMDb ratings.")
    else:
        print("  ⚠ No IMDb ratings found — keeping existing imdb_rating/imdb_votes columns if present.")

    # Recalculate best_* for ALL rows
    df_catalog["vote_average"] = pd.to_numeric(df_catalog.get("vote_average"), errors="coerce")
    df_catalog["vote_count"]   = pd.to_numeric(df_catalog.get("vote_count"), errors="coerce").fillna(0)
    df_catalog["imdb_rating"]  = pd.to_numeric(df_catalog["imdb_rating"], errors="coerce")
    df_catalog["imdb_votes"]   = pd.to_numeric(df_catalog["imdb_votes"], errors="coerce").fillna(0).astype(int)
    df_catalog["best_rating"]  = df_catalog["imdb_rating"].fillna(df_catalog["vote_average"])
    df_catalog["best_votes"]   = df_catalog["imdb_votes"].where(
        df_catalog["imdb_votes"] > 0, df_catalog["vote_count"]
    )
    df_catalog["best_votes"]   = pd.to_numeric(df_catalog["best_votes"], errors="coerce").fillna(0).astype(int)

    # Rebuild movieDoc for ALL rows (old + new, with fresh IMDb data)
    print("  Rebuilding movieDoc for ALL rows (this may take a few minutes)...")
    df_catalog["movieDoc"] = df_catalog.apply(lambda row: build_moviedoc(row.to_dict()), axis=1)

    # ── 6. Save catalog CSV in-place ──────────────────────────────
    df_catalog.to_csv(paths["catalog"], index=False)
    print(f"\n  ✅ Saved updated catalog → {paths['catalog']}")
    print(f"     {len(df_catalog):,} rows total | "
          f"{df_catalog['imdb_rating'].notna().sum():,} with IMDb ratings")

    elapsed = time.time() - t_start
    print(f"\n  Total time so far: {elapsed:.1f}s")
    print("\n  ✅ STEP 1–5 COMPLETE. Run the next cell to encode and rebuild FAISS.\n")

    # Expose paths for the next cell
    return paths


paths = main()


In [6]:

# ━━━━━━━━━━━━━━━━━━━━━  STEP 6: FULL FAISS REBUILD FROM SCRATCH  ━━━━━━━━━━━━
# Reads the updated catalog CSV, encodes ALL rows with BGE-M3, builds a brand-
# new IndexFlatIP + IndexIDMap2 from scratch, and writes it to disk.
# Estimated time on Colab L4: ~4-6 hours for ~1.37M rows at batch_size=2048.

import pandas as pd
import numpy as np
import faiss
import torch
import json
import time
from pathlib import Path
from sentence_transformers import SentenceTransformer

# paths was set by main() above; re-detect if the cell is run standalone
if "paths" not in dir() or paths is None:
    paths = detect_paths()

t0 = time.time()

print(f"{'═'*60}")
print(" FULL FAISS REBUILD — encoding ALL catalog rows")
print(f"{'═'*60}")

# 1. Load catalog
print(f"\n Loading catalog: {paths['catalog']}")
catalog = pd.read_csv(paths["catalog"], low_memory=False)
catalog["id"] = pd.to_numeric(catalog["id"], errors="coerce")
catalog = catalog.dropna(subset=["id"])
catalog["id"] = catalog["id"].astype(int)
print(f"  Total rows: {len(catalog):,}")

# 2. Filter embeddable rows (must have a real Plot in movieDoc, length >= 80)
embeddable_mask = (
    catalog["movieDoc"].notna()
    & catalog["movieDoc"].str.contains("Plot:", na=False)
    & (catalog["movieDoc"].str.len() >= 80)
)
embed_df = catalog.loc[embeddable_mask, ["id", "movieDoc"]].copy()
print(f"  Embeddable rows: {len(embed_df):,} "
      f"({len(catalog) - len(embed_df):,} skipped — missing Plot or too short)")

# 3. Load BGE-M3
device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
print(f"\n Loading BAAI/bge-m3 on {device}...")
model = SentenceTransformer(MODEL_ID, device=device)
emb_dim = model.get_sentence_embedding_dimension() or 1024
print(f"  Embedding dim: {emb_dim}")

# 4. Encode in batches with progress logging
texts   = embed_df["movieDoc"].tolist()
ids_arr = embed_df["id"].to_numpy(dtype=np.int64)

print(f"\n Encoding {len(texts):,} texts (batch={ENCODE_BATCH}) ...")
encode_start = time.time()

with torch.no_grad():
    vectors = model.encode(
        texts,
        batch_size=ENCODE_BATCH,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

# Double normalise (belt + suspenders — bge-m3 does it but cosine via IP needs unit vectors)
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
norms[norms == 0] = 1.0
vectors /= norms

encode_elapsed = time.time() - encode_start
print(f"  Encoded {len(vectors):,} vectors in {encode_elapsed:.1f}s  |  shape: {vectors.shape}")

# 5. Build a brand-new FAISS index from scratch
print(f"\n Building new FAISS IndexFlatIP + IndexIDMap2 (dim={emb_dim})...")
base_index = faiss.IndexFlatIP(emb_dim)
index      = faiss.IndexIDMap2(base_index)
index.add_with_ids(vectors, ids_arr)
print(f"  Index total: {index.ntotal:,} vectors")

# 6. Write FAISS to disk
paths["out_dir"].mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(paths["faiss"]))
print(f"\n  ✅ FAISS saved → {paths['faiss']}")

# 7. Write manifest
manifest = {
    "build": {
        "timestamp_utc":   pd.Timestamp.utcnow().isoformat(),
        "fetch_window":    f"{FETCH_START_DATE} → {date.today().isoformat()}",
        "catalog_rows":    int(len(catalog)),
        "embedded_rows":   int(len(embed_df)),
        "skipped_rows":    int(len(catalog) - len(embed_df)),
    },
    "embedding": {
        "model":           MODEL_ID,
        "dim":             int(emb_dim),
        "normalize":       True,
        "batch_size":      ENCODE_BATCH,
        "encode_elapsed_s": round(encode_elapsed, 1),
    },
    "faiss": {
        "type":            "IndexFlatIP + IndexIDMap2",
        "ntotal":          int(index.ntotal),
        "path":            "outputs/tmdb/bge/tmdb_bge_m3_flatip.faiss",
    },
    "catalog": {
        "path":            "Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv",
        "rows":            int(len(catalog)),
        "imdb_matched":    int(catalog["imdb_rating"].notna().sum()),
    },
}
paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"  ✅ Manifest saved → {paths['manifest']}")

total_elapsed = time.time() - t0
print(f"\n{'═'*60}")
print(f"  FAISS REBUILD DONE in {total_elapsed / 60:.1f} min")
print(f"  Vectors: {index.ntotal:,}  |  Dim: {emb_dim}")
print(f"{'═'*60}")
print("\n  ✅ STEP 6 COMPLETE. Run the next cell to upload to Hugging Face.\n")


In [7]:

# ━━━━━━━━━━━━━━━━━━━━━  STEP 7: UPLOAD TO HUGGING FACE  ━━━━━━━━━━━━━━━━━━━━
import os
from huggingface_hub import HfApi, login
from google.colab import userdata

# paths was set by main() above; re-detect if the cell is run standalone
if "paths" not in dir() or paths is None:
    paths = detect_paths()

try:
    hf_token = userdata.get("HF_TOKEN")
    login(hf_token)
except Exception as e:
    print("❌ Please add your Hugging Face write token to Colab Secrets as 'HF_TOKEN'.")
    raise e

api       = HfApi()
repo_id   = "ml8r/cinematch"
repo_type = "dataset"

upload_tasks = [
    {
        "local":  str(paths["catalog"]),
        "remote": "Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv",
        "msg":    f"Full catalog refresh: {FETCH_START_DATE}→today, fresh IMDb ratings, rebuilt movieDocs",
    },
    {
        "local":  str(paths["faiss"]),
        "remote": "outputs/tmdb/bge/tmdb_bge_m3_flatip.faiss",
        "msg":    "Full FAISS rebuild from scratch (BGE-M3, IndexFlatIP+IndexIDMap2)",
    },
    {
        "local":  str(paths["manifest"]),
        "remote": "outputs/tmdb/bge/tmdb_bge_m3_build_manifest.json",
        "msg":    "Update build manifest",
    },
]

print(f"\nInitiating upload to {repo_id}...\n")

for task in upload_tasks:
    local_path  = task["local"]
    remote_path = task["remote"]
    if os.path.exists(local_path):
        size_gb = os.path.getsize(local_path) / 1e9
        print(f"⬆️  Uploading: {os.path.basename(local_path)}  ({size_gb:.2f} GB)")
        print(f"    → {remote_path}")
        try:
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=remote_path,
                repo_id=repo_id,
                repo_type=repo_type,
                commit_message=task["msg"],
            )
            print("    ✅ Done.\n")
        except Exception as e:
            print(f"    ❌ Failed: {e}\n")
    else:
        print(f"⚠️  File not found (skipping): {local_path}\n")

print("🎉 Push to Hugging Face complete!")
